In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Interp"
x_col, y_col = "x", "y"
df = pd.read_excel(file_path, sheet_name=sheet_name)
x = df[x_col].to_numpy(dtype=float)
y = df[y_col].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "x_new": 2.5  # float: 需要插值的位置
}

def lagrange_interpolate(x, y, x_new):
    n, res = len(x), 0.0
    for i in range(n):
        term = y[i]
        for j in range(n):
            if i != j:
                term *= (x_new - x[j]) / (x[i] - x[j])
        res += term
    return res

def newton_interpolate(x, y, x_new):
    n = len(x)
    coef = y.astype(float).copy()
    for j in range(1, n):
        coef[j:n] = (coef[j:n] - coef[j-1:n-1]) / (x[j:n] - x[0:n-j])
    res, prod = coef[0], 1.0
    for i in range(1, n):
        prod *= (x_new - x[i-1])
        res += coef[i]*prod
    return res

print(lagrange_interpolate(x, y, params["x_new"]))
print(newton_interpolate(x, y, params["x_new"]))


In [ ]:
"""
拉格朗日插值法、牛顿插值法

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "拉格朗日插值法、牛顿插值法.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import KroghInterpolator, lagrange



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
X_COLUMN = "x"  # TODO: 请填写[自变量列名]，说明：数值型且不能重复。
Y_COLUMN = "y"  # TODO: 请填写[因变量列名]，说明：与 x 一一对应。
TODO_X_NEW = [1.5, 2.5]  # TODO: 请填写[待插值点]，说明：建议位于已知 x 范围内。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def lagrange_interpolate(x, y, x_new):
    return lagrange(x, y)(x_new)


def newton_interpolate(x, y, x_new):
    # scipy 的 KroghInterpolator 可用于 Newton 型多项式插值。
    return KroghInterpolator(x, y)(x_new)


def run_model(data: pd.DataFrame) -> None:
    x = data[X_COLUMN].to_numpy(dtype=float)
    y = data[Y_COLUMN].to_numpy(dtype=float)
    x_new = np.array(TODO_X_NEW, dtype=float)
    result = pd.DataFrame({
        X_COLUMN: x_new,
        "拉格朗日插值": lagrange_interpolate(x, y, x_new),
        "牛顿插值": newton_interpolate(x, y, x_new),
    })
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
